## Data Exploration /cleaning

In [2]:
import pandas as pd

# ============================
# 1. Load datasets
# ============================
train_df = pd.read_csv("../data/raw/train.csv")
eval_df = pd.read_csv("../data/raw/eval.csv")
metros = pd.read_csv("../data/raw/usmetros.csv")

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows

In [3]:
print(train_df.shape)
print(eval_df.shape)

(585244, 39)
(149424, 39)


In [4]:
train_df['city_full'].value_counts().head()

city_full
New York-Newark-Jersey City       78020
Chicago-Naperville-Elgin          35344
Los Angeles-Long Beach-Anaheim    33840
Philadelphia-Camden-Wilmington    31396
DC_Metro                          29516
Name: count, dtype: int64

### Map cities to Lat/Long
- The goal is to use Lattitude and longitude instead of cities for our ML models

In [5]:
# ============================
# 2. Fix city name mismatches
# ============================
city_mapping = {
    'Las Vegas-Henderson-Paradise': 'Las Vegas-Henderson-North Las Vegas',
    'Denver-Aurora-Lakewood': 'Denver-Aurora-Centennial',
    'Houston-The Woodlands-Sugar Land': 'Houston-Pasadena-The Woodlands',
    'Austin-Round Rock-Georgetown': 'Austin-Round Rock-San Marcos',
    'Miami-Fort Lauderdale-Pompano Beach': 'Miami-Fort Lauderdale-West Palm Beach',
    'San Francisco-Oakland-Berkeley': 'San Francisco-Oakland-Fremont',
    'DC_Metro': 'Washington-Arlington-Alexandria',
    'Atlanta-Sandy Springs-Alpharetta': 'Atlanta-Sandy Springs-Roswell'
}

In [ ]:
def clean_and_merge(df: pd.DataFrame) -> pd.DataFrame:
    """Apply city name fixes, merge lat/lng from metros, drop dup col."""

    # 1. Corriger les noms de villes
    df["city_full"] = df["city_full"].replace(city_mapping)

    # 2. Ajouter latitude et longitude depuis metros
    df = df.merge(
        metros[["metro_full", "lat", "lng"]],
        how="left",
        left_on="city_full",
        right_on="metro_full"
    )

    # 3. Supprimer la colonne dupliquée
    df.drop(columns=["metro_full"], inplace=True)

    # 4. Vérifier les villes qui n'ont pas trouvé de correspondance
    missing = df[df["lat"].isnull()]["city_full"].unique()

    if len(missing) > 0:
        print("⚠️ Still missing lat/lng for:", missing)
    else:
        print("✅ All cities matched with metros dataset.")

    return df

In [7]:
print(metros.head())

   metro_fips        metro  metro_ascii                          metro_full  \
0       35620     New York     New York  New York-Newark-Jersey City, NY-NJ   
1       31080  Los Angeles  Los Angeles  Los Angeles-Long Beach-Anaheim, CA   
2       16980      Chicago      Chicago     Chicago-Naperville-Elgin, IL-IN   
3       19100       Dallas       Dallas     Dallas-Fort Worth-Arlington, TX   
4       26420      Houston      Houston  Houston-Pasadena-The Woodlands, TX   

   county_name  county_fips state_id  state_name      lat       lng  \
0      Suffolk        36103       NY    New York  40.7222  -74.0225   
1  Los Angeles         6037       CA  California  34.2215 -118.1494   
2         Cook        17031       IL    Illinois  41.6675  -87.9597   
3       Denton        48121       TX       Texas  32.8495  -96.9704   
4       Harris        48201       TX       Texas  29.8422  -95.3855   

   population  
0    19940274  
1    12927614  
2     9406924  
3     8344032  
4     7796182  
